# ML Network Intrusion Detector

**Project:** Binary classification of network traffic as Benign or Attack using Scikit-learn and CIC-IDS2017.

This notebook is a learning prototype. It trains on labeled network-flow records and evaluates predictions on a held-out test split. It does not monitor live network traffic.

## Before running
1. Download the CIC-IDS2017 machine-learning CSV files from the official Canadian Institute for Cybersecurity page: https://www.unb.ca/cic/datasets/ids-2017.html
2. Extract the CSV files into a folder named `data` beside this notebook.
3. Install dependencies using `pip install -r requirements.txt`.
4. Run the cells from top to bottom.


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

RANDOM_STATE = 42
DATA_DIR = Path('data')
MAX_ROWS = 150_000  # Increase if your computer has enough memory.
warnings.filterwarnings('ignore', category=RuntimeWarning)
print('Libraries loaded. Data directory:', DATA_DIR.resolve())


## 1. Load the dataset
The official dataset contains several CSV files and is large. This loader reads available CSVs, standardizes column names, and limits the training sample for modest computers. Make sure the CSVs are inside `data/`.


In [ ]:
csv_files = sorted(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(
        "No CSV files found. Download/extract CIC-IDS2017 CSV files into a 'data' folder beside this notebook."
    )

frames = []
for file in csv_files:
    try:
        frame = pd.read_csv(file, low_memory=False, encoding_errors='replace')
        frame.columns = [str(c).strip() for c in frame.columns]
        # CIC-IDS2017 files sometimes contain a leading space in the label column.
        frames.append(frame)
        print(f'Loaded {file.name}: {frame.shape[0]:,} rows, {frame.shape[1]} columns')
    except Exception as exc:
        print(f'Skipped {file.name}: {exc}')

if not frames:
    raise ValueError('CSV files were found, but none could be read.')
data = pd.concat(frames, ignore_index=True, sort=False)
data.columns = [str(c).strip() for c in data.columns]
print('\nCombined dataset shape:', data.shape)
display(data.head())


## 2. Clean and prepare the data
The label column contains `BENIGN` for normal traffic and attack names for malicious traffic. We convert this into a binary target: `0 = Benign`, `1 = Attack`. Identifier-like columns are removed because they may not generalize and can create misleading results.


In [ ]:
# Find the label column case-insensitively.
label_candidates = [c for c in data.columns if c.strip().lower() == 'label']
if not label_candidates:
    raise ValueError(f'Could not find a Label column. Available columns: {list(data.columns)}')
label_col = label_candidates[0]

data = data.dropna(subset=[label_col]).copy()
labels = data[label_col].astype(str).str.strip()
y = (labels.str.upper() != 'BENIGN').astype('int8')
X = data.drop(columns=[label_col])

# Remove fields that identify a flow/host or can act as metadata rather than reusable features.
identifier_columns = [c for c in X.columns if c.strip().lower() in {
    'flow id', 'source ip', 'destination ip', 'timestamp', 'src ip', 'dst ip'
}]
X = X.drop(columns=identifier_columns, errors='ignore')

# Convert every feature to numeric; invalid values become missing.
X = X.apply(pd.to_numeric, errors='coerce')
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna(axis=1, how='all')
X = X.fillna(X.median(numeric_only=True)).fillna(0)

# Remove constant columns and duplicate feature names if present.
X = X.loc[:, X.nunique(dropna=False) > 1]
X = X.loc[:, ~X.columns.duplicated()]

if X.empty:
    raise ValueError('No usable numeric features remain after cleaning.')

# Randomly sample for manageable training time, retaining class representation where possible.
if len(X) > MAX_ROWS:
    sampled_idx = data.sample(n=MAX_ROWS, random_state=RANDOM_STATE).index
    X = X.loc[sampled_idx]
    y = y.loc[sampled_idx]

print('Features:', X.shape[1])
print('Rows used:', len(X))
print('Class counts (0=Benign, 1=Attack):')
display(y.value_counts().rename(index={0: 'Benign', 1: 'Attack'}))
if y.nunique() < 2:
    raise ValueError('The loaded sample has only one class. Include CSV files containing both benign and attack traffic.')


## 3. Split data and train the model
A stratified 80/20 split helps preserve the class proportions in both sets. The Random Forest model is a practical baseline for tabular network-flow data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = RandomForestClassifier(
    n_estimators=150,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    min_samples_leaf=2,
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print('Model training complete.')
print('Training rows:', len(X_train), '| Test rows:', len(X_test))


## 4. Evaluate the model
Accuracy alone can hide missed attacks when classes are imbalanced. Review precision, recall, F1-score, and the confusion matrix together. Results depend on the files included and the sample size.


In [ ]:
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision (Attack)': precision_score(y_test, y_pred, zero_division=0),
    'Recall (Attack)': recall_score(y_test, y_pred, zero_division=0),
    'F1-score (Attack)': f1_score(y_test, y_pred, zero_division=0),
}
for name, value in metrics.items():
    print(f'{name}: {value:.4f}')

print('\nDetailed classification report:')
print(classification_report(y_test, y_pred, target_names=['Benign', 'Attack'], zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Benign', 'Predicted Attack'],
            yticklabels=['Actual Benign', 'Actual Attack'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()


## 5. Inspect important features
Feature importance is a model-specific estimate of which features contributed most to the Random Forest's decisions. It does not prove causation.


In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
display(importance.rename('Importance').to_frame())
plt.figure(figsize=(9, 6))
importance.sort_values().plot(kind='barh')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 6. Try a sample prediction
This uses a row from the held-out test set to demonstrate the prediction format. It is not a live network monitor.


In [ ]:
sample = X_test.iloc[[0]]
prediction = int(model.predict(sample)[0])
probability = model.predict_proba(sample)[0][1]
print('Prediction:', 'ATTACK' if prediction == 1 else 'BENIGN')
print(f'Model-estimated attack probability: {probability:.3f}')


## Conclusion and limitations
- This project demonstrates supervised binary classification of labeled network-flow data.
- The reported scores apply to the chosen dataset files, preprocessing, sample, and split; they do not guarantee performance on a live network.
- CIC-IDS2017 is a benchmark dataset from 2017. Traffic patterns and attack techniques change over time.
- Dataset artifacts, class imbalance, and possible overlap between related flows can inflate evaluation scores. A stronger follow-up would use time-based or source-separated evaluation and external validation.
- Use this as a defensive learning prototype, not as the sole security control in a production environment.
